# Упражнение: Разложение «как Prophet»: тренд с изломом, Фурье и праздники МНК

## Задание

Prophet раскладывает ряд в сумму y(t) = g(t) + s(t) + h(t) + ε(t). Библиотеки prophet у нас нет —
соберите ту же аддитивную модель как линейную регрессию на специально построенных признаках.

Файлы: `input.csv` — дневной ряд (`ds, y`, с 2023-01-01 по 2024-12-31),
`holidays.csv` — праздники в формате Prophet (`holiday, ds, lower_window, upper_window`).

Время t — число дней от первой даты ряда (t = 0 для 2023-01-01). Признаки:

1. **Тренд g(t)** — кусочно-линейный с одной точкой излома t_c = 2024-01-01:
   признаки `t` и `max(0, t − t_c)` (наклон после излома меняется на коэффициент второго признака).
2. **Недельная сезонность** — ряд Фурье с P = 7, N = 3: sin(2πkt/7), cos(2πkt/7), k = 1..3.
3. **Годовая сезонность** — P = 365.25, N = 2: sin(2πkt/365.25), cos(2πkt/365.25), k = 1..2.
4. **Праздники h(t)** — по одной индикаторной колонке на каждое имя `holiday`: 1, если дата попадает
   в окно [ds + lower_window, ds + upper_window] **какого-либо** вхождения этого праздника, иначе 0.
   (У `new_year` окно −1…+1: 31 декабря, 1 и 2 января.)

Обучите `LinearRegression()` (свободный член — часть тренда) на датах **до 2024-11-01**,
предскажите период 2024-11-01 … 2024-12-31 и разложите прогноз на компоненты:

- `trend` = свободный член + вклад двух признаков тренда;
- `weekly`, `yearly` — вклад соответствующих Фурье-признаков (сумма коэффициент × признак);
- `holidays` — вклад праздничных колонок;
- `yhat` = trend + weekly + yearly + holidays.

## Подсказки

- Это ровно то, что показывает `plot_components` в Prophet: прогноз — сумма слагаемых, и каждое
  можно посчитать отдельно, умножив свою группу признаков на свои коэффициенты.
- Праздник 2025-01-01 лежит за пределами ряда, но его окно (lower_window = −1) накрывает 2024-12-31 —
  поэтому в таблице праздников есть и будущие даты. Признаки праздников должны быть известны на период прогноза.
- Соберите матрицу X колонками по группам (тренд / weekly / yearly / праздники), обучите модель на
  `X[ds < '2024-11-01']`, затем `X[cols] @ coef[cols]` для каждой группы; проверьте, что сумма
  компонент совпадает с `model.predict(X)`.

## Ожидаемый формат выходного файла

CSV с колонками `ds, trend, weekly, yearly, holidays, yhat` — 61 строка (2024-11-01 … 2024-12-31 по порядку),
`ds` в формате `YYYY-MM-DD`; числа округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/prophet_time_series
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.